In [1]:
import pandas as pd

DATA_PATH = r"Z:\MINGLE\Data\Melanoma\23_10_11_Melanoma_Marker_Cell_Neighborhood.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} cells")
print(df.columns.tolist())
display(df.head())

Loaded 5,019,159 cells
['Unnamed: 0', 'CCR7', 'CD103', 'CD11b', 'CD11c', 'CD138', 'CD15', 'CD163', 'CD20', 'CD206', 'CD21', 'CD25', 'CD3', 'CD31', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD68', 'CD73', 'CD8', 'CLEC9A', 'CXCL10', 'CXCL13', 'CXCL9', 'CXCR3', 'CXCR5', 'CollagenIV', 'EOMES', 'EZH2', 'FAP', 'FoxP3', 'GATA3', 'GranzymeB', 'HLA-DR', 'ICOS', 'IL10', 'IRF8', 'Ki67', 'LAG3', 'LDH-A', 'MHC-I', 'MMP9', 'Melan-A', 'PD-1', 'PD-L1', 'PTEN', 'Podoplanin', 'SOX10/S100', 'TCF1/7', 'Tbet', 'Vimentin', 'aSMA', 'bCatenin', 'pERK1/2', 'DAPI', 'cellid', 'donor', 'CCL5', 'Cell_Type_Common', 'Cell_Type_Sub', 'CD38', 'filename', 'region', 'x', 'y', 'Cell_Type', 'Overall_Cell_Type', 'Neighborhood']


,Unnamed: 0,CCR7,CD103,CD11b,CD11c,CD138,CD15,CD163,CD20,CD206,...,Cell_Type_Common,Cell_Type_Sub,CD38,filename,region,x,y,Cell_Type,Overall_Cell_Type,Neighborhood
0,0,-0.362926,-0.729925,-0.150820,-0.363121,-0.193573,-0.376392,-0.277710,-0.096556,-0.326673,...,Stromal,CD73+ Stromal,NaN,21_06_23_reg001.tsv,2,3981,47146,CD73+ Stromal,Stromal,Stromal Enriched
1,1,-0.542319,-0.723530,-0.193650,-0.359325,-0.199382,-0.394698,-0.219857,-0.126534,-0.333797,...,Stromal,CD73+ Stromal,NaN,21_06_23_reg001.tsv,2,5286,39373,CD73+ Stromal,Stromal,Stromal Enriched
2,2,-0.444684,-0.497128,-0.329550,-0.292091,-0.201171,-0.202878,-0.309667,-0.135984,-0.347450,...,Endothelial,Endothelial,NaN,21_06_23_reg001.tsv,2,22899,42306,Endothelial,Stromal,Vasculature
3,3,-0.829088,-1.629941,-0.341772,-0.380618,-0.255592,-0.419650,-0.115570,-0.118611,0.245775,...,Stromal,CD73+ Stromal,NaN,21_06_23_reg001.tsv,2,5747,45479,CD73+ Stromal,Stromal,Stromal Enriched
4,4,0.111211,-0.134976,3.659562,0.540194,-0.181382,5.446553,-0.212632,-0.125781,-0.246451,...,Neutrophil,Neutrophil,NaN,21_06_23_reg001.tsv,2,17460,49393,Neutrophil,Immune,Neutrophil Enriched


In [2]:
df.columns.unique()

Index(['Unnamed: 0', 'CCR7', 'CD103', 'CD11b', 'CD11c', 'CD138', 'CD15',
       'CD163', 'CD20', 'CD206', 'CD21', 'CD25', 'CD3', 'CD31', 'CD4', 'CD45',
       'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD68', 'CD73', 'CD8', 'CLEC9A',
       'CXCL10', 'CXCL13', 'CXCL9', 'CXCR3', 'CXCR5', 'CollagenIV', 'EOMES',
       'EZH2', 'FAP', 'FoxP3', 'GATA3', 'GranzymeB', 'HLA-DR', 'ICOS', 'IL10',
       'IRF8', 'Ki67', 'LAG3', 'LDH-A', 'MHC-I', 'MMP9', 'Melan-A', 'PD-1',
       'PD-L1', 'PTEN', 'Podoplanin', 'SOX10/S100', 'TCF1/7', 'Tbet',
       'Vimentin', 'aSMA', 'bCatenin', 'pERK1/2', 'DAPI', 'cellid', 'donor',
       'CCL5', 'Cell_Type_Common', 'Cell_Type_Sub', 'CD38', 'filename',
       'region', 'x', 'y', 'Cell_Type', 'Overall_Cell_Type', 'Neighborhood'],
      dtype='object')

In [53]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors

# Change these names if your DataFrame uses different columns.
REGION_COL = "filename"
NEIGHBORHOOD_COL = "Neighborhood"
X_COL = "x"
Y_COL = "y"

NUM_NEIGHBORS = 10
MIN_INSTANCE_SIZE = 50  # Inclusive: retain patches with >= 10 cells

required_cols = [REGION_COL, NEIGHBORHOOD_COL, X_COL, Y_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise KeyError(f"Missing columns in df: {missing_cols}")

# Give each input row a unique positional ID, even if df's index has duplicates.
analysis_df = df[required_cols].dropna().reset_index(drop=True).copy()
analysis_df[[X_COL, Y_COL]] = analysis_df[[X_COL, Y_COL]].apply(
    pd.to_numeric, errors="raise"
)

if analysis_df.empty:
    raise ValueError("No cells remain after removing rows with missing values.")

if not np.isfinite(analysis_df[[X_COL, Y_COL]].to_numpy()).all():
    raise ValueError("Coordinates must be finite.")

if NUM_NEIGHBORS < 1 or MIN_INSTANCE_SIZE < 1:
    raise ValueError("NUM_NEIGHBORS and MIN_INSTANCE_SIZE must be positive.")

analysis_df["cell_id"] = np.arange(len(analysis_df), dtype=np.int64)

print(f"Cells analyzed: {len(analysis_df):,}")
print(f"Regions: {analysis_df[REGION_COL].nunique():,}")
print(f"Neighborhoods: {analysis_df[NEIGHBORHOOD_COL].nunique():,}")
print(f"Rows excluded due to missing required values: {len(df) - len(analysis_df):,}")

Cells analyzed: 5,019,159
Regions: 21
Neighborhoods: 16
Rows excluded due to missing required values: 0


In [54]:
patch_rows = []
patch_cell_rows = []
next_patch_id = 0

for region_id, region in analysis_df.groupby(
    REGION_COL, sort=False, observed=True
):
    coordinates = region[[X_COL, Y_COL]].to_numpy()
    n_region_cells = len(region)

    if n_region_cells == 1:
        region_graph = csr_matrix((1, 1), dtype=np.int8)
    else:
        k = min(NUM_NEIGHBORS, n_region_cells - 1)

        nn = NearestNeighbors(n_neighbors=k)
        nn.fit(coordinates)

        # With X=None, sklearn finds neighbors of the fitted points
        # and excludes each point itself.
        region_graph = nn.kneighbors_graph(
            X=None,
            n_neighbors=k,
            mode="connectivity",
        )

        # Treat a connection in either direction as an undirected edge.
        region_graph = region_graph.maximum(region_graph.T).tocsr()

    neighborhood_values = region[NEIGHBORHOOD_COL].to_numpy()
    cell_ids = region["cell_id"].to_numpy()

    for neighborhood in pd.unique(neighborhood_values):
        positions = np.flatnonzero(neighborhood_values == neighborhood)

        neighborhood_graph = region_graph[positions, :][:, positions]

        _, component_labels = connected_components(
            neighborhood_graph,
            directed=False,
        )

        component_sizes = np.bincount(component_labels)

        for component_id, patch_size in enumerate(component_sizes):
            if patch_size < MIN_INSTANCE_SIZE:
                continue

            member_positions = positions[component_labels == component_id]

            patch_rows.append({
                "patch_id": next_patch_id,
                REGION_COL: region_id,
                NEIGHBORHOOD_COL: neighborhood,
                "n_cells": int(patch_size),
            })

            patch_cell_rows.extend(
                (next_patch_id, int(cell_id))
                for cell_id in cell_ids[member_positions]
            )

            next_patch_id += 1

patches_df = pd.DataFrame(
    patch_rows,
    columns=["patch_id", REGION_COL, NEIGHBORHOOD_COL, "n_cells"],
)

patch_cells_df = pd.DataFrame(
    patch_cell_rows,
    columns=["patch_id", "cell_id"],
)

if patches_df.empty:
    raise ValueError(
        "No patches passed MIN_INSTANCE_SIZE. Check your labels, "
        "coordinates, graph connectivity, and size cutoff."
    )

print(f"Retained patches: {len(patches_df):,}")
print(f"Cells in retained patches: {len(patch_cells_df):,}")
print(
    f"Cells outside retained patches: "
    f"{len(analysis_df) - len(patch_cells_df):,}"
)

display(patches_df.head())

Retained patches: 8,233
Cells in retained patches: 3,158,168
Cells outside retained patches: 1,860,991


,patch_id,filename,Neighborhood,n_cells
0,0,21_06_23_reg001.tsv,Stromal Enriched,25685
1,1,21_06_23_reg001.tsv,Stromal Enriched,52
2,2,21_06_23_reg001.tsv,Stromal Enriched,213
3,3,21_06_23_reg001.tsv,Stromal Enriched,83
4,4,21_06_23_reg001.tsv,Stromal Enriched,191


In [55]:
from scipy.sparse import triu

# Require this many spatial graph connections between two patches
# before calling them adjacent.
MIN_CROSS_PATCH_CONNECTIONS = 1

# Map every analyzed cell to its retained patch.
# Cells in components below MIN_INSTANCE_SIZE remain -1.
cell_to_patch = np.full(len(analysis_df), -1, dtype=np.int64)

cell_to_patch[
    patch_cells_df["cell_id"].to_numpy(dtype=np.int64)
] = patch_cells_df["patch_id"].to_numpy(dtype=np.int64)

adjacency_parts = []

for region_id, region in analysis_df.groupby(
    REGION_COL, sort=False, observed=True
):
    cell_ids = region["cell_id"].to_numpy(dtype=np.int64)
    coordinates = region[[X_COL, Y_COL]].to_numpy()
    n_region_cells = len(region)

    if n_region_cells < 2:
        continue

    k = min(NUM_NEIGHBORS, n_region_cells - 1)

    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(coordinates)

    graph = nn.kneighbors_graph(
        X=None,
        n_neighbors=k,
        mode="connectivity",
    )
    graph = graph.maximum(graph.T).tocsr()

    # Keep each undirected cell-to-cell connection once.
    edges = triu(graph, k=1).tocoo()

    patch_a = cell_to_patch[cell_ids[edges.row]]
    patch_b = cell_to_patch[cell_ids[edges.col]]

    # Both cells must belong to retained, different patches.
    valid = (patch_a >= 0) & (patch_b >= 0) & (patch_a != patch_b)

    if not np.any(valid):
        continue

    pairs = pd.DataFrame({
        "patch_id_a": np.minimum(patch_a[valid], patch_b[valid]),
        "patch_id_b": np.maximum(patch_a[valid], patch_b[valid]),
    })

    adjacency_parts.append(pairs)

if not adjacency_parts:
    raise ValueError(
        "No connections were found between retained patches. "
        "Try inspecting patch sizes or graph connectivity."
    )

# Count the cell-to-cell connections for each pair of patches.
patch_adjacency = (
    pd.concat(adjacency_parts, ignore_index=True)
    .groupby(["patch_id_a", "patch_id_b"], as_index=False)
    .size()
    .rename(columns={"size": "n_cross_patch_connections"})
)

patch_adjacency = patch_adjacency.loc[
    patch_adjacency["n_cross_patch_connections"]
    >= MIN_CROSS_PATCH_CONNECTIONS
].copy()

if patch_adjacency.empty:
    raise ValueError(
        "No patch pairs pass MIN_CROSS_PATCH_CONNECTIONS."
    )

# Retain only patches touching at least one other retained patch.
adjacent_patch_ids = np.unique(
    patch_adjacency[["patch_id_a", "patch_id_b"]].to_numpy().ravel()
)

patches_all_df = patches_df
patch_cells_all_df = patch_cells_df

patches_df = patches_all_df.loc[
    patches_all_df["patch_id"].isin(adjacent_patch_ids)
].copy()

patch_cells_df = patch_cells_all_df.loc[
    patch_cells_all_df["patch_id"].isin(adjacent_patch_ids)
].copy()

print(f"All retained patches before adjacency filter: {len(patches_all_df):,}")
print(f"Adjacent patch pairs: {len(patch_adjacency):,}")
print(f"Patches adjacent to another retained patch: {len(patches_df):,}")
print(f"Cells in adjacent patches: {len(patch_cells_df):,}")

display(patch_adjacency.head())
display(patches_df.head())

All retained patches before adjacency filter: 8,233
Adjacent patch pairs: 11,802
Patches adjacent to another retained patch: 7,941
Cells in adjacent patches: 3,131,397


,patch_id_a,patch_id_b,n_cross_patch_connections
0,0,130,133
1,0,140,199
2,0,146,19
3,0,147,41
4,0,148,169


,patch_id,filename,Neighborhood,n_cells
0,0,21_06_23_reg001.tsv,Stromal Enriched,25685
1,1,21_06_23_reg001.tsv,Stromal Enriched,52
2,2,21_06_23_reg001.tsv,Stromal Enriched,213
3,3,21_06_23_reg001.tsv,Stromal Enriched,83
4,4,21_06_23_reg001.tsv,Stromal Enriched,191


In [56]:
neighborhood_summary = (
    patches_df
    .groupby(NEIGHBORHOOD_COL, observed=True)
    .agg(
        n_patches=("patch_id", "size"),
        n_regions=(REGION_COL, "nunique"),
        total_cells_in_patches=("n_cells", "sum"),
        mean_cells_per_patch=("n_cells", "mean"),
        median_cells_per_patch=("n_cells", "median"),
    )
    .reset_index()
)

# Show neighborhoods with no retained patches as missing rather than zero.
all_neighborhoods = pd.DataFrame({
    NEIGHBORHOOD_COL: pd.unique(analysis_df[NEIGHBORHOOD_COL])
})

neighborhood_summary = all_neighborhoods.merge(
    neighborhood_summary,
    on=NEIGHBORHOOD_COL,
    how="left",
)

average_of_neighborhood_averages = (
    neighborhood_summary["mean_cells_per_patch"].mean()
)

print(
    "Average cells per patch, averaged equally across neighborhoods: "
    f"{average_of_neighborhood_averages:.2f}"
)
print(
    "Neighborhoods with retained patches: "
    f"{neighborhood_summary['mean_cells_per_patch'].notna().sum():,}"
)
print(
    "Neighborhoods without retained patches: "
    f"{neighborhood_summary['mean_cells_per_patch'].isna().sum():,}"
)

display(neighborhood_summary)

Average cells per patch, averaged equally across neighborhoods: 387.39
Neighborhoods with retained patches: 16
Neighborhoods without retained patches: 0


,Neighborhood,n_patches,n_regions,total_cells_in_patches,mean_cells_per_patch,median_cells_per_patch
0,Stromal Enriched,504,21,279834,555.226190,96.0
1,Vasculature,64,5,22409,350.140625,79.5
2,Neutrophil Enriched,54,9,50069,927.203704,125.0
3,Macrophage Enriched Immune,791,19,123097,155.621997,77.0
4,Vasculature & Immune,560,21,54096,96.600000,70.0
5,Productive T cell & Tumor,397,20,230357,580.244332,100.0
6,Proliferating Tumor,1302,21,657334,504.864823,94.0
7,Inflamed Tumor,1110,20,531121,478.487387,95.0
8,Tumor & Immune,236,5,21647,91.724576,73.0
9,Epithelial/Skin Appendages,389,21,354162,910.442159,171.0


In [57]:
CELL_DIAMETER = 1.0  # Set to a physical value only if you know it.
DIAMETER_UNIT = "cell diameters"
AREA_UNIT = "cell-area units"

def add_circle_geometry(frame, count_col):
    result = frame.copy()
    n = result[count_col].astype(float)

    result["estimated_area"] = n * CELL_DIAMETER**2
    result["estimated_radius"] = CELL_DIAMETER * np.sqrt(n / np.pi)
    result["estimated_diameter"] = 2 * result["estimated_radius"]

    return result

# Geometry for every observed patch.
patches_with_geometry = add_circle_geometry(patches_df, "n_cells")

# Geometry of a hypothetical patch with each neighborhood's mean size.
neighborhood_geometry = add_circle_geometry(
    neighborhood_summary.dropna(subset=["mean_cells_per_patch"]),
    "mean_cells_per_patch",
)

# Geometry of a hypothetical patch with the overall mean of neighborhood means.
overall_geometry = add_circle_geometry(
    pd.DataFrame({
        "mean_cells_per_patch": [average_of_neighborhood_averages]
    }),
    "mean_cells_per_patch",
)

print(f"Area units: {AREA_UNIT}")
print(f"Diameter units: {DIAMETER_UNIT}")

display(
    neighborhood_geometry[
        [
            NEIGHBORHOOD_COL,
            "n_patches",
            "mean_cells_per_patch",
            "estimated_area",
            "estimated_diameter",
        ]
    ]
)

display(overall_geometry)

Area units: cell-area units
Diameter units: cell diameters


,Neighborhood,n_patches,mean_cells_per_patch,estimated_area,estimated_diameter
0,Stromal Enriched,504,555.226190,555.226190,26.588267
1,Vasculature,64,350.140625,350.140625,21.114282
2,Neutrophil Enriched,54,927.203704,927.203704,34.359168
3,Macrophage Enriched Immune,791,155.621997,155.621997,14.076366
4,Vasculature & Immune,560,96.600000,96.600000,11.090308
5,Productive T cell & Tumor,397,580.244332,580.244332,27.180692
6,Proliferating Tumor,1302,504.864823,504.864823,25.353774
7,Inflamed Tumor,1110,478.487387,478.487387,24.682566
8,Tumor & Immune,236,91.724576,91.724576,10.806820
9,Epithelial/Skin Appendages,389,910.442159,910.442159,34.047187


,mean_cells_per_patch,estimated_area,estimated_radius,estimated_diameter
0,387.385561,387.385561,11.104443,22.208886


In [58]:
def estimated_border_fraction(n_cells, width_in_cell_diameters):
    """
    Circular-patch estimate of the fraction within 'width' of the edge.
    n_cells can be a number or array; width is measured inward from
    the patch boundary in cell-diameter units.
    """
    n = np.asarray(n_cells, dtype=float)
    w = float(width_in_cell_diameters)

    if np.any(n <= 0) or w < 0:
        raise ValueError("Patch sizes must be positive and width nonnegative.")

    inner_radius_fraction = np.clip(
        1 - w * np.sqrt(np.pi / n),
        0,
        1,
    )
    return 1 - inner_radius_fraction**2


border_rows = []

for width in range(1, 6):
    # The requested two-stage calculation: average patch size within
    # each neighborhood, then average those neighborhood means.
    fraction_from_overall_mean = float(
        estimated_border_fraction(
            average_of_neighborhood_averages,
            width,
        )
    )

    # Calculate the fraction for each actual retained patch.
    patch_fractions = estimated_border_fraction(
        patches_df["n_cells"].to_numpy(),
        width,
    )

    # Mean predicted fraction within each neighborhood, followed by
    # an equal-weight mean across neighborhoods.
    per_neighborhood_fraction = (
        patches_df.assign(predicted_border_fraction=patch_fractions)
        .groupby(NEIGHBORHOOD_COL, observed=True)[
            "predicted_border_fraction"
        ]
        .mean()
    )
    equal_neighborhood_fraction = float(
        per_neighborhood_fraction.mean()
    )

    # Weight each patch's predicted fraction by its number of cells.
    # This estimates the fraction of cells in retained patches
    # that could lie in the border band.
    predicted_border_cells = float(
        np.sum(patches_df["n_cells"].to_numpy() * patch_fractions)
    )
    total_retained_cells = int(patches_df["n_cells"].sum())
    cell_weighted_fraction = (
        predicted_border_cells / total_retained_cells
    )

    border_rows.append({
        "width_cell_diameters": width,
        "mean_cells_per_patch_equal_neighborhoods":
            average_of_neighborhood_averages,
        "border_fraction_from_mean_patch":
            fraction_from_overall_mean,
        "border_percent_from_mean_patch":
            100 * fraction_from_overall_mean,
        "nonborder_percent_from_mean_patch":
            100 * (1 - fraction_from_overall_mean),
        "border_percent_equal_neighborhoods":
            100 * equal_neighborhood_fraction,
        "border_percent_cell_weighted":
            100 * cell_weighted_fraction,
        "nonborder_percent_cell_weighted":
            100 * (1 - cell_weighted_fraction),
    })

border_width_summary = pd.DataFrame(border_rows)

display(border_width_summary.round(2))

,width_cell_diameters,mean_cells_per_patch_equal_neighborhoods,border_fraction_from_mean_patch,border_percent_from_mean_patch,nonborder_percent_from_mean_patch,border_percent_equal_neighborhoods,border_percent_cell_weighted,nonborder_percent_cell_weighted
0,1,387.39,0.17,17.20,82.80,31.80,10.92,89.08
1,2,387.39,0.33,32.78,67.22,56.71,20.25,79.75
2,3,387.39,0.47,46.73,53.27,74.72,27.99,72.01
3,4,387.39,0.59,59.07,40.93,85.84,34.14,65.86
4,5,387.39,0.70,69.78,30.22,91.16,38.83,61.17
